# Double Pendulum: Empirical Loss Saturation Profiler (1B Pilot Run)

This notebook runs empirical loss saturation profiling across a representative 15-model matrix (varying width $d_{\text{model}} \in \{32, 64, 128\}$, lookahead horizon $k \in \{1, \text{mid}, \text{full}\}$, and step resolution $n \in \{10, 20\}$) to identify the exact token budget where next-token and lookahead cross-entropy loss plateaus.

### Kaggle Setup Instructions:
1. **Accelerator**: Select **GPU T4 x2** in the right-side Settings panel (`cuda:0` and `cuda:1` run concurrently via `ThreadPoolExecutor`).
2. **Internet**: Toggle **Internet: On** in the right-side Settings panel.
3. **Execution Mode**:
   - For quick verification (~30 seconds): Run the **Smoke Dry-Run** cell.
   - For the full 1B token run: Click **Save Version** -> **Save & Run All (Commit)** to execute headless in the background (up to 12 hours) so browser disconnections will not interrupt training.

In [ ]:
# --- 1. CLONE & SETUP REPOSITORY ---
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
REPO_URL = 'https://github.com/ManjotSingh08x/Belief-in-physics.git'
REPO_BRANCH = 'double_pendulum_system'

if IS_KAGGLE:
    target = Path('/kaggle/working/Belief-in-physics')
    if not target.exists():
        print(f"Cloning branch '{REPO_BRANCH}' from {REPO_URL}...")
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(target)], check=True)
    else:
        print(f"Repository already present at {target}")
    os.chdir(target)
    if str(target) not in sys.path:
        sys.path.insert(0, str(target))
else:
    repo_root = Path.cwd()
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

print(f"Current Working Directory: {Path.cwd()}")

# Check CUDA Devices
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} GPU(s):")
for i in range(n_gpus):
    print(f"  [cuda:{i}] {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB)")

In [ ]:
# --- 2. SMOKE DRY-RUN (VERIFY IN ~20 SECONDS) ---
# Runs 2 models for 125 steps with logging every 5 steps
# Verifies data streaming, loss descent, gradient accumulation, and plot generation
!python scripts/loss_saturation_check.py --smoke

In [ ]:
# --- 3. FULL 15-MODEL SATURATION PROFILER (DUAL-GPU) ---
# Automatically distributes 15 models across cuda:0 and cuda:1 in parallel
# Target: 1,000,000,000 tokens per model (or adjust e.g. --tokens 500000000)
!python scripts/loss_saturation_check.py --tokens 1000000000 --batch-size 1024 --log-every 20

In [ ]:
# --- 4. DISPLAY SATURATION DASHBOARD & MILESTONES ---
from IPython.display import Image, display
import pandas as pd
from pathlib import Path

is_kaggle = Path('/kaggle').exists()
out_dir = Path('/kaggle/working/results_saturation') if is_kaggle else Path('experiments/results_saturation')

# Display 4-panel publication-quality analytical figure
fig_file = out_dir / 'loss_saturation_analysis.png'
if fig_file.exists():
    print("\n--- 4-PANEL LOSS SATURATION ANALYSIS ---")
    display(Image(filename=str(fig_file)))
else:
    print(f"Figure not found at {fig_file}")

# Display summary table with saturation milestones (90%, 95%, 98%, 99% drop)
summary_file = out_dir / 'loss_saturation_summary.csv'
if summary_file.exists():
    print("\n--- SATURATION MILESTONES TABLE ---")
    df_summary = pd.read_csv(summary_file)
    display(df_summary)
else:
    print(f"Summary CSV not found at {summary_file}")

### Downloading Artifacts
The profiler automatically compresses all checkpoints, CSVs, and figures into:
`/kaggle/working/loss_saturation_results.zip`

To download:
1. Look at the right sidebar in Kaggle under the **Output** tab.
2. Locate `loss_saturation_results.zip`.
3. Click the three dots `...` next to the file and select **Download**.